In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge

# 1. Load data
train = pd.read_csv("/kaggle/input/datasets/kuladeeproy/notebookdata/training_data.csv")
test = pd.read_csv("/kaggle/input/datasets/kuladeeproy/notebookdata/test_data_X.csv")
precip = pd.read_csv("/kaggle/input/datasets/kuladeeproy/notebookdata/precipitation_data.csv")


# 2. Standardize columns
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()
precip.columns = precip.columns.str.strip()

train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])
precip["date"] = pd.to_datetime(precip["date"])

train = train.sort_values("date").reset_index(drop=True)
test = test.sort_values("date").reset_index(drop=True)
precip = precip.sort_values("date").reset_index(drop=True)


# 3. Rename precipitation column
precip_cols = [c for c in precip.columns if c != "date"]
if len(precip_cols) != 1:
    raise ValueError(f"Expected one precipitation column besides date, found: {precip_cols}")

precip_col = precip_cols[0]
precip = precip.rename(columns={precip_col: "precip"})


# 4. Merge precipitation
train = train.merge(precip, on="date", how="left")
test = test.merge(precip, on="date", how="left")

train["precip"] = train["precip"].fillna(0)
test["precip"] = test["precip"].fillna(0)

print(train.head())
print(test.head())
print(train.shape, test.shape)

        date  sm_surface  precip
0 2023-01-01    0.156469    2.75
1 2023-01-02    0.155497    0.01
2 2023-01-03    0.155756    6.94
3 2023-01-04    0.154115    4.57
4 2023-01-05    0.145088    1.20
        date  precip
0 2024-10-01    1.42
1 2024-10-02    0.00
2 2024-10-03    0.01
3 2024-10-04    0.19
4 2024-10-05    0.00
(639, 3) (93, 2)


In [2]:
# 5. Build delta-feature matrix

def make_delta_features(df):
    df = df.copy().sort_values("date").reset_index(drop=True)

    # calendar
    df["dayofyear"] = df["date"].dt.dayofyear
    df["month"]     = df["date"].dt.month
    df["dayofweek"] = df["date"].dt.dayofweek
    df["sin_doy"]   = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
    df["cos_doy"]   = np.cos(2 * np.pi * df["dayofyear"] / 365.25)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)

    # lagged soil moisture levels
    for lag in [1, 2, 3, 5, 7, 10, 14, 21, 30, 45, 60]:
        df[f"y_lag_{lag}"] = df["sm_surface"].shift(lag)

    # momentum features
    df["diff_1"]  = df["sm_surface"].shift(1) - df["sm_surface"].shift(2)
    df["diff_2"]  = df["sm_surface"].shift(2) - df["sm_surface"].shift(3)
    df["diff_3"]  = df["sm_surface"].shift(3) - df["sm_surface"].shift(4)
    df["diff_7"]  = df["sm_surface"].shift(1) - df["sm_surface"].shift(8)
    df["diff_14"] = df["sm_surface"].shift(1) - df["sm_surface"].shift(15)
    df["diff_30"] = df["sm_surface"].shift(1) - df["sm_surface"].shift(31)

    # rolling stats from past only
    s = df["sm_surface"].shift(1)
    for win in [3, 7, 14, 30, 60]:
        df[f"roll_mean_{win}"] = s.rolling(win, min_periods=1).mean()
        df[f"roll_std_{win}"]  = s.rolling(win, min_periods=1).std()
        df[f"roll_min_{win}"]  = s.rolling(win, min_periods=1).min()
        df[f"roll_max_{win}"]  = s.rolling(win, min_periods=1).max()

    # z-score vs recent window
    df["sm_zscore_14"] = (s - df["roll_mean_14"]) / (df["roll_std_14"] + 1e-8)
    df["sm_zscore_30"] = (s - df["roll_mean_30"]) / (df["roll_std_30"] + 1e-8)

    # precipitation lags
    for lag in [0, 1, 2, 3, 5, 7, 10, 14]:
        df[f"precip_lag_{lag}"] = df["precip"].shift(lag)

    # precipitation rolling features
    p = df["precip"].shift(1)
    for win in [3, 7, 14, 30, 60]:
        df[f"precip_sum_{win}"]  = p.rolling(win, min_periods=1).sum()
        df[f"precip_mean_{win}"] = p.rolling(win, min_periods=1).mean()
        df[f"precip_max_{win}"]  = p.rolling(win, min_periods=1).max()

    # Interaction: precip effect modulated by current moisture level
    df["precip_x_sm_lag1"] = df["precip_lag_1"] * df["y_lag_1"]
    df["precip_sum7_x_sm"] = df["precip_sum_7"] * df["y_lag_1"]

    # target = next daily change
    df["delta_target"] = df["sm_surface"] - df["sm_surface"].shift(1)

    return df

train_delta = make_delta_features(train).dropna().reset_index(drop=True)

delta_features = [
    c for c in train_delta.columns
    if c not in ["date", "sm_surface", "delta_target"]
]

X_delta = train_delta[delta_features]
y_delta = train_delta["delta_target"]

print(X_delta.shape, y_delta.shape)
print(delta_features)

(579, 72) (579,)
['precip', 'dayofyear', 'month', 'dayofweek', 'sin_doy', 'cos_doy', 'sin_month', 'cos_month', 'y_lag_1', 'y_lag_2', 'y_lag_3', 'y_lag_5', 'y_lag_7', 'y_lag_10', 'y_lag_14', 'y_lag_21', 'y_lag_30', 'y_lag_45', 'y_lag_60', 'diff_1', 'diff_2', 'diff_3', 'diff_7', 'diff_14', 'diff_30', 'roll_mean_3', 'roll_std_3', 'roll_min_3', 'roll_max_3', 'roll_mean_7', 'roll_std_7', 'roll_min_7', 'roll_max_7', 'roll_mean_14', 'roll_std_14', 'roll_min_14', 'roll_max_14', 'roll_mean_30', 'roll_std_30', 'roll_min_30', 'roll_max_30', 'roll_mean_60', 'roll_std_60', 'roll_min_60', 'roll_max_60', 'sm_zscore_14', 'sm_zscore_30', 'precip_lag_0', 'precip_lag_1', 'precip_lag_2', 'precip_lag_3', 'precip_lag_5', 'precip_lag_7', 'precip_lag_10', 'precip_lag_14', 'precip_sum_3', 'precip_mean_3', 'precip_max_3', 'precip_sum_7', 'precip_mean_7', 'precip_max_7', 'precip_sum_14', 'precip_mean_14', 'precip_max_14', 'precip_sum_30', 'precip_mean_30', 'precip_max_30', 'precip_sum_60', 'precip_mean_60', 'pre

In [3]:
# 6. Time-series CV for DELTA model

import lightgbm as lgb

tscv = TimeSeriesSplit(n_splits=5)

delta_models = {
    "ridge": Ridge(alpha=0.05),
    "rf": RandomForestRegressor(
        n_estimators=700, max_depth=10, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ),
    "etr": ExtraTreesRegressor(
        n_estimators=1200, max_depth=12, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ),
    "gbr": GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.02, max_depth=2, random_state=42
    ),
    "lgbm": lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=6,
        num_leaves=31,
        min_child_samples=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
}

delta_cv_scores = {name: [] for name in delta_models}

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_delta), 1):
    X_train, X_val = X_delta.iloc[train_idx], X_delta.iloc[val_idx]
    y_train, y_val = y_delta.iloc[train_idx], y_delta.iloc[val_idx]

    for name, model in delta_models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        delta_cv_scores[name].append(rmse)
        print(f"Fold {fold} | {name} delta RMSE = {rmse:.8f}")

print("\nAverage delta CV RMSE")
delta_avg_scores = {}
for name, scores in delta_cv_scores.items():
    avg = np.mean(scores)
    delta_avg_scores[name] = avg
    print(f"  {name}: {avg:.8f}")

Fold 1 | ridge delta RMSE = 0.03576631
Fold 1 | rf delta RMSE = 0.01453086
Fold 1 | etr delta RMSE = 0.01372209
Fold 1 | gbr delta RMSE = 0.01699461
Fold 1 | lgbm delta RMSE = 0.01546044
Fold 2 | ridge delta RMSE = 0.01977559
Fold 2 | rf delta RMSE = 0.01188225
Fold 2 | etr delta RMSE = 0.01163751
Fold 2 | gbr delta RMSE = 0.01213973
Fold 2 | lgbm delta RMSE = 0.01169852
Fold 3 | ridge delta RMSE = 0.01296814
Fold 3 | rf delta RMSE = 0.01164981
Fold 3 | etr delta RMSE = 0.01129064
Fold 3 | gbr delta RMSE = 0.01250321
Fold 3 | lgbm delta RMSE = 0.01184929
Fold 4 | ridge delta RMSE = 0.01177783
Fold 4 | rf delta RMSE = 0.01288640
Fold 4 | etr delta RMSE = 0.01242252
Fold 4 | gbr delta RMSE = 0.01203612
Fold 4 | lgbm delta RMSE = 0.01259519
Fold 5 | ridge delta RMSE = 0.01064570
Fold 5 | rf delta RMSE = 0.00906295
Fold 5 | etr delta RMSE = 0.00861194
Fold 5 | gbr delta RMSE = 0.00842733
Fold 5 | lgbm delta RMSE = 0.00826684

Average delta CV RMSE
  ridge: 0.01818671
  rf: 0.01200245
  etr

In [4]:
# 7. Fit final delta models on ALL training data + compute ensemble weights


# Inverse-RMSE weighting: better CV score → higher weight
inv_rmse  = {name: 1.0 / score for name, score in delta_avg_scores.items()}
total_inv = sum(inv_rmse.values())
delta_weights = {name: inv_rmse[name] / total_inv for name in inv_rmse}

print("Ensemble weights (inverse-RMSE):")
for name, w in delta_weights.items():
    print(f"  {name}: {w:.4f}")

# Fit every model on the FULL training set
fitted_delta_models = {}
for name, model in delta_models.items():
    model.fit(X_delta, y_delta)
    fitted_delta_models[name] = model
    print(f"  Fitted {name} on {len(X_delta)} training samples")

print("\nAll models fitted and ready for forecasting.")

Ensemble weights (inverse-RMSE):
  ridge: 0.1413
  rf: 0.2142
  etr: 0.2228
  gbr: 0.2070
  lgbm: 0.2147
  Fitted ridge on 579 training samples
  Fitted rf on 579 training samples
  Fitted etr on 579 training samples
  Fitted gbr on 579 training samples
  Fitted lgbm on 579 training samples

All models fitted and ready for forecasting.


In [5]:
# 8. Final recursive forecast — tuned for 93-day horizon


history = train.copy().sort_values("date").reset_index(drop=True)
preds   = []

# Build a same-day-of-year lookup from 2023 training data (seasonal anchor)
history["dayofyear"] = history["date"].dt.dayofyear
seasonal_anchor = history.groupby("dayofyear")["sm_surface"].mean().to_dict()

# Weights tuned for 93-step recursive horizon
# delta_w kept moderate to prevent error compounding
persist_w   = 0.52
delta_w     = 0.32
trend_w     = 0.08
seasonal_w  = 0.06
mean_w      = 0.02

for i in range(len(test)):
    next_row = test.iloc[[i]].copy()
    next_row["sm_surface"] = np.nan

    temp      = pd.concat([history, next_row], ignore_index=True)
    temp_feat = make_delta_features(temp)

    last_row  = temp_feat.iloc[[-1]].copy()
    X_curr    = last_row[delta_features].ffill().bfill().fillna(0)

    # ---- ensemble delta prediction ----
    pred_delta = 0.0
    for name, model in fitted_delta_models.items():
        pred_delta += delta_weights[name] * model.predict(X_curr)[0]

    last_y          = history["sm_surface"].iloc[-1]
    pred_from_delta = last_y + pred_delta

    # ---- damped trend ----
    if len(history) >= 4:
        d1 = history["sm_surface"].iloc[-1] - history["sm_surface"].iloc[-2]
        d2 = history["sm_surface"].iloc[-2] - history["sm_surface"].iloc[-3]
        d3 = history["sm_surface"].iloc[-3] - history["sm_surface"].iloc[-4]
        damped_trend = 0.55 * d1 + 0.30 * d2 + 0.15 * d3
    elif len(history) >= 3:
        d1 = history["sm_surface"].iloc[-1] - history["sm_surface"].iloc[-2]
        d2 = history["sm_surface"].iloc[-2] - history["sm_surface"].iloc[-3]
        damped_trend = 0.75 * d1 + 0.25 * d2
    else:
        damped_trend = 0.0

    trend_pred = last_y + damped_trend

    # ---- seasonal anchor from same day-of-year in training ----
    next_doy     = next_row["date"].dt.dayofyear.values[0]
    # average over ±3 days window to smooth anchor
    anchor_vals  = [seasonal_anchor.get((next_doy + d) % 366, np.nan) for d in range(-3, 4)]
    anchor_vals  = [v for v in anchor_vals if not np.isnan(v)]
    seasonal_val = np.mean(anchor_vals) if anchor_vals else last_y

    # ---- recent mean ----
    recent_mean_14 = history["sm_surface"].tail(14).mean()

    # ---- main blend ----
    pred_level = (
        persist_w  * last_y
        + delta_w  * pred_from_delta
        + trend_w  * trend_pred
        + seasonal_w * seasonal_val
    )

    # ---- pull toward recent mean ----
    pred_level = (1 - mean_w) * pred_level + mean_w * recent_mean_14

    # ---- calibrated precipitation response ----
    if "precip" in history.columns and len(history) >= 3:
        rain_adj = (
            0.000045 * history["precip"].iloc[-1]
            + 0.000020 * history["precip"].iloc[-2]
            + 0.000010 * history["precip"].iloc[-3]
        )
        pred_level += rain_adj

    # ---- clipping: tighter than before, but not as extreme as original ----
    recent_low_30  = history["sm_surface"].tail(30).min()
    recent_high_30 = history["sm_surface"].tail(30).max()
    pred_level = np.clip(pred_level, recent_low_30 - 0.002, recent_high_30 + 0.002)

    preds.append(pred_level)

    next_row = next_row.drop(columns=["dayofyear"], errors="ignore")
    next_row["sm_surface"] = pred_level
    history = pd.concat([history, next_row], ignore_index=True)

test_preds = np.array(preds)

print(pd.Series(test_preds).head(10))
print("Shape:", test_preds.shape)
print("Std:  ", np.std(test_preds))
print("Min:  ", np.min(test_preds))
print("Max:  ", np.max(test_preds))

0    0.109863
1    0.109640
2    0.108911
3    0.109082
4    0.109265
5    0.109842
6    0.110849
7    0.111096
8    0.111035
9    0.110898
dtype: float64
Shape: (93,)
Std:   0.013836338581786397
Min:   0.1078445588298969
Max:   0.15176417811107368


In [8]:
# =========================
# 9. Build submission
# =========================

submission = pd.DataFrame({
    "rowId": np.arange(len(test_preds)),
    "label": test_preds
})

submission.to_csv("submission9.csv", index=False)

print(submission.head())
print(submission.shape)

   rowId     label
0      0  0.109863
1      1  0.109640
2      2  0.108911
3      3  0.109082
4      4  0.109265
(93, 2)
